# 🪐 ALTAIR: Astro-Financial & Explainable Glassbox Model (EBM) Analysis

This notebook provides an interactive testing playground for the Altair backend AI analysis, Astro Growth Score calculations, and training Explainable Boosting Machines (EBM) as Glassbox models.

## 1. Setup & Environment Verification

Ensure you have `interpret` (for EBM models), `pandas`, `numpy`, and `matplotlib` installed in your environment. If not, you can install them by running:
```bash
pip install interpret jupyter
```

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root is in path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

print("Project Root set to:", project_root)
print("Current directory:", os.getcwd())

## 2. Ingest Corporate Registry & Strike Map Data

Let's load the company natal registration metadata and the computed financial metrics from the strike map.

In [ ]:
registry_path = os.path.join(project_root, "astro", "company", "company_natal_registry.csv")
strike_map_path = os.path.join(project_root, "data", "processed", "GLOBAL_STRIKE_MAP_2026.csv")

if os.path.exists(registry_path):
    df_reg = pd.read_csv(registry_path)
    print(f"[+] Loaded Natal Registry: {len(df_reg)} companies.")
    display(df_reg.head(5))
else:
    print("[!] Registry not found. Check path:", registry_path)

if os.path.exists(strike_map_path):
    df_strike = pd.read_csv(strike_map_path)
    print(f"[+] Loaded Strike Map: {len(df_strike)} companies.")
    display(df_strike.head(5))
else:
    print("[!] Strike Map not found. Run a full audit first.")

## 3. Explainable Boosting Machines (EBM) as Glassbox Models

An **Explainable Boosting Machine (EBM)** is a tree-based generalized additive model (GAM) that is fully interpretable ("glassbox"). It is built on the principle that the predictions are a sum of individual feature functions:

$$F(x) = \beta_0 + \sum f_i(x_i) + \sum f_{ij}(x_i, x_j)$$

Unlike black-box models (like neural networks or deep forests), which are explained via post-hoc methods (like SHAP), EBMs have exact explainability because we can plot the exact function $f_i$ for each feature. EBMs are highly accurate, often matching the performance of XGBoost while remaining fully transparent.

In [ ]:
try:
    from interpret.glassbox import ExplainableBoostingRegressor
    from interpret import show
    
    print("[+] Interpret library is available!")
    
    if os.path.exists(strike_map_path):
        # Prepare features (e.g. predicting strike score based on AVS and PE ratio)
        df = pd.read_csv(strike_map_path).dropna(subset=["avs_score", "pe_ratio", "astra_strike_score"])
        X = df[["avs_score", "pe_ratio"]]
        y = df["astra_strike_score"]
        
        # Initialize EBM Regressor
        ebm = ExplainableBoostingRegressor(random_state=42)
        ebm.fit(X, y)
        print("[+] Glassbox EBM Model trained successfully!")
        
        # Retrieve and print global explanations
        ebm_global = ebm.explain_global()
        print("\n=== Feature Importance scores ===")
        for name, score in zip(ebm_global.data()["names"], ebm_global.data()["scores"]):
            print(f"  {name: <12}: {score:.4f}")
            
    else:
        print("[!] Strike map not available for training EBM.")
except ImportError:
    print("[!] 'interpret' package is not installed. Install via: pip install interpret")

## 4. Test the Astro-Financial Compiler

Let's test the `analyze_company_alpha` logic from our middleware to verify that we can fetch and compile company data.

In [ ]:
try:
    # Import our custom middleware analyzer
    from astro.company.alpha_analyzer import analyze_company_alpha
    
    ticker = "RELIANCE.NS"  # Switch to "AAPL" or others to test
    result = analyze_company_alpha(ticker)
    
    print(f"\n[+] Compiled interpreted call for {ticker}:")
    print(json.dumps(result, indent=2))
except Exception as e:
    print("[!] Error running alpha analysis:", e)